In [1]:
import pandas as pd
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import postgres_connection

settings = Settings.from_environment()

QUERY = """
SELECT
    s.source_record_id,
    s.experiment_id,
    s.shot_index,
    e.distance,
    e.rounds,
    e.center_row,
    e.center_col,
    e.detector_count,
    s.detector_event_count,
    s.detector_bits,
    s.actual_observable_flip,
    d.decoder_id,
    sp.predicted_flip
FROM shot s
JOIN experiment e ON e.experiment_id = s.experiment_id
JOIN shot_prediction sp ON sp.shot_id = s.shot_id
JOIN decoder d ON d.decoder_id = sp.decoder_id
"""

with postgres_connection(settings) as conn:
    raw = pd.read_sql(QUERY, conn)

print(raw.shape)
raw.head()

/tmp/ipykernel_359/2801791064.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  raw = pd.read_sql(QUERY, conn)


(1000000, 13)


,source_record_id,experiment_id,shot_index,distance,rounds,center_row,center_col,detector_count,detector_event_count,detector_bits,actual_observable_flip,decoder_id,predicted_flip
0,google_shot_bf9253e1d330,google_surface_code_bX_d3_r25_center_3_5,13,3,25,3,5,200,47,"b'\x004\x08\x04\r\x08\x08\x00\x00 ""\x06\x06\x0...",True,belief_matching,False
1,google_shot_bf9253e1d330,google_surface_code_bX_d3_r25_center_3_5,13,3,25,3,5,200,47,"b'\x004\x08\x04\r\x08\x08\x00\x00 ""\x06\x06\x0...",True,correlated_matching,False
2,google_shot_bf9253e1d330,google_surface_code_bX_d3_r25_center_3_5,13,3,25,3,5,200,47,"b'\x004\x08\x04\r\x08\x08\x00\x00 ""\x06\x06\x0...",True,pymatching,False
3,google_shot_bf9253e1d330,google_surface_code_bX_d3_r25_center_3_5,13,3,25,3,5,200,47,"b'\x004\x08\x04\r\x08\x08\x00\x00 ""\x06\x06\x0...",True,tensor_network_contraction,True
4,google_shot_d667d5515ed5,google_surface_code_bX_d3_r25_center_3_5,14,3,25,3,5,200,33,b' \x07\x01\x10\x00\x04\x06\x02\x00 \x10\x10\x...,False,belief_matching,True


In [2]:
shot_columns = [
    "source_record_id", "experiment_id", "shot_index", "distance", "rounds",
    "center_row", "center_col", "detector_count", "detector_event_count",
    "detector_bits", "actual_observable_flip",
]

df = raw[shot_columns].drop_duplicates("source_record_id").reset_index(drop=True)

decoder_names = {
    "belief_matching": "belief_matching_prediction",
    "correlated_matching": "correlated_matching_prediction",
    "pymatching": "pymatching_prediction",
    "tensor_network_contraction": "tensor_network_contraction_prediction",
}

for decoder_id, column_name in decoder_names.items():
    decoder_rows = raw[raw["decoder_id"] == decoder_id][["source_record_id", "predicted_flip"]]
    decoder_rows = decoder_rows.rename(columns={"predicted_flip": column_name})
    df = df.merge(decoder_rows, on="source_record_id")

df.head()

,source_record_id,experiment_id,shot_index,distance,rounds,center_row,center_col,detector_count,detector_event_count,detector_bits,actual_observable_flip,belief_matching_prediction,correlated_matching_prediction,pymatching_prediction,tensor_network_contraction_prediction
0,google_shot_bf9253e1d330,google_surface_code_bX_d3_r25_center_3_5,13,3,25,3,5,200,47,"b'\x004\x08\x04\r\x08\x08\x00\x00 ""\x06\x06\x0...",True,False,False,False,True
1,google_shot_d667d5515ed5,google_surface_code_bX_d3_r25_center_3_5,14,3,25,3,5,200,33,b' \x07\x01\x10\x00\x04\x06\x02\x00 \x10\x10\x...,False,True,True,True,True
2,google_shot_c99b8b3e1a3c,google_surface_code_bX_d3_r25_center_3_5,16,3,25,3,5,200,26,b'\x00\x81\x00\x00\x86\x01\x00\x00\x90\x00\t\x...,True,False,True,True,False
3,google_shot_26ab84f73220,google_surface_code_bX_d3_r25_center_3_5,18,3,25,3,5,200,27,b'\x00\x00\x00\t\x08\x00\x04\x05\x03\x00\x00\x...,False,True,False,False,False
4,google_shot_f040f83012b1,google_surface_code_bX_d3_r25_center_3_5,36,3,25,3,5,200,40,b'\x00\x00\x00\x00\x00\x00 \x07\x03\xa0!A\xa6\...,True,True,False,False,True


In [3]:
# Cell 2 — generate example_id and the data_split column
import hashlib
from quantum_lake_student.ml import google_data_split

example_ids = []
data_splits = []

# key = str(df["experiment_id"][0]) + ":" + str(df["shot_index"][0])
# print(key)
# key.encode()
# hashlib.md5(key.encode()).hexdigest()

for i in range(len(df)):
    experiment_id = df["experiment_id"][i]
    shot_index = df["shot_index"][i]

    key = str(experiment_id) + ":" + str(shot_index)
    hashed = hashlib.md5(key.encode()).hexdigest()
    example_ids.append("ml_" + hashed[:12])

    data_splits.append(google_data_split(shot_index))

df["example_id"] = example_ids
df["data_split"] = data_splits

df[["example_id", "data_split"]].head()
# print(df["data_split"].value_counts())

,example_id,data_split
0,ml_2a88952d8637,test
1,ml_f253ab2fa507,train
2,ml_27e57f145cd3,train
3,ml_af2b7be50ac9,validation
4,ml_6700547a12a4,train


In [4]:
duplicate_count = len(df) - len(df["example_id"].drop_duplicates())
if duplicate_count > 0:
    raise ValueError(f"example_id is not unique, {duplicate_count} duplicates found")

bad_length_rows = []
for i in range(len(df)):
    detector_bits = df["detector_bits"][i]
    detector_count = df["detector_count"][i]
    expected_length = (detector_count + 7) // 8
    if len(detector_bits) != expected_length:
        bad_length_rows.append(i)

if len(bad_length_rows) > 0:
    raise ValueError(f"{len(bad_length_rows)} rows have wrong detector_bits length")

found_splits = set()
for i in range(len(df)):
    found_splits.add(df["data_split"][i])

required_splits = {"train", "validation", "test"}
if found_splits != required_splits:
    missing = required_splits - found_splits
    raise ValueError(f"missing data_split values: {missing}")

print("All checks passed")

All checks passed


In [5]:
# Cell 4 — select final column order/shape and write out
ml_columns = [
    "example_id", "experiment_id", "shot_index", "distance", "rounds",
    "center_row", "center_col", "detector_count", "detector_event_count",
    "detector_bits", "belief_matching_prediction", "correlated_matching_prediction",
    "pymatching_prediction", "tensor_network_contraction_prediction",
    "actual_observable_flip", "data_split",
]
df_ml = df[ml_columns].reset_index(drop=True)

import os
os.makedirs("results/part2/ml", exist_ok=True)
df_ml.to_parquet("results/part2/ml/ml_google_decoder_example.parquet", index=False)

df_ml.shape

(250000, 16)

In [6]:
from quantum_lake_student.ml import google_meta_model_input

def get_feature_row(df1, h):
    row_dict = {}
    for c_name in df1.columns:
        row_dict[c_name] = df1[c_name][h]
    return google_meta_model_input(row_dict)

train_df = df_ml[df_ml["data_split"] == "train"].reset_index(drop=True)
val_df = df_ml[df_ml["data_split"] == "validation"].reset_index(drop=True)
test_df = df_ml[df_ml["data_split"] == "test"].reset_index(drop=True)

X_train = []
for i in range(len(train_df)):
    X_train.append(get_feature_row(train_df, i))

y_train = []
for i in range(len(train_df)):
    y_train.append(int(train_df["actual_observable_flip"][i]))

X_test = []
for i in range(len(test_df)):
    X_test.append(get_feature_row(test_df, i))

y_test = []
for i in range(len(test_df)):
    y_test.append(int(test_df["actual_observable_flip"][i]))

print(len(X_train), len(X_test))
X_train[:5]

100000 125000


[(0.165, 1, 1, 1, 1),
 (0.13, 0, 1, 1, 0),
 (0.2, 1, 0, 0, 1),
 (0.135, 0, 0, 0, 1),
 (0.115, 1, 1, 1, 1)]

In [7]:
train_df.columns

Index(['example_id', 'experiment_id', 'shot_index', 'distance', 'rounds',
       'center_row', 'center_col', 'detector_count', 'detector_event_count',
       'detector_bits', 'belief_matching_prediction',
       'correlated_matching_prediction', 'pymatching_prediction',
       'tensor_network_contraction_prediction', 'actual_observable_flip',
       'data_split'],
      dtype='object')

In [8]:
row_dict = {}
for c_name in train_df.columns:
    row_dict[c_name] = train_df[c_name][1]
row_dict

{'example_id': 'ml_27e57f145cd3',
 'experiment_id': 'google_surface_code_bX_d3_r25_center_3_5',
 'shot_index': np.int64(16),
 'distance': np.int64(3),
 'rounds': np.int64(25),
 'center_row': np.int64(3),
 'center_col': np.int64(5),
 'detector_count': np.int64(200),
 'detector_event_count': np.int64(26),
 'detector_bits': b'\x00\x81\x00\x00\x86\x01\x00\x00\x90\x00\t\x00\x00\x00@\x08\x08\x02\x02\x00\x80\x8d\x01\x86\x06',
 'belief_matching_prediction': np.False_,
 'correlated_matching_prediction': np.True_,
 'pymatching_prediction': np.True_,
 'tensor_network_contraction_prediction': np.False_,
 'actual_observable_flip': np.True_,
 'data_split': 'train'}

In [9]:
google_meta_model_input(row_dict)

(0.13, 0, 1, 1, 0)

In [10]:
y_train[:5]

[0, 1, 1, 1, 1]

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, brier_score_loss

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("accuracy:", accuracy_score(y_test, y_pred))
print("balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
print("brier score:", brier_score_loss(y_test, y_prob))

accuracy: 0.60384
balanced accuracy: 0.6037238346478266
brier score: 0.2349210747317171


In [14]:
for col in ["belief_matching_prediction", "correlated_matching_prediction",
            "pymatching_prediction", "tensor_network_contraction_prediction"]:
    decoder_acc = accuracy_score(y_test, test_df[col].astype(int))
    print(f"{col}: {decoder_acc:.4f}")

print(f"combined model: {accuracy_score(y_test, y_pred):.4f}")

belief_matching_prediction: 0.5990
correlated_matching_prediction: 0.5779
pymatching_prediction: 0.5600
tensor_network_contraction_prediction: 0.6036
combined model: 0.6025
